# AccentShift — Seed-VC Fine-Tuning on Kaggle

**Before running:**
1. Enable GPU: Settings (right panel) → Accelerator → **GPU T4 x2** or P100
2. Enable Internet: Settings → Internet → On
3. Run all cells top to bottom

Expected total time: ~2–3 hours on T4, ~3–5 hours on P100.
Download checkpoint from Files panel (right) after Cell 7 completes.

In [ ]:
# Cell 1: Verify GPU
!nvidia-smi

In [ ]:
%%bash
# Cell 2: Clone repo
git clone --depth 1 https://github.com/nischal2805/AccentShift.git /kaggle/working/AccentShift
cd /kaggle/working/AccentShift && git checkout div
echo "Repo cloned, branch: $(git branch --show-current)"

In [ ]:
%%bash
# Cell 3: Install pipeline deps (torch already on Kaggle)
pip install -q \
    "librosa>=0.10.2" soundfile pyloudnorm pyworld silero-vad \
    transformers jiwer speechbrain scikit-learn joblib pyyaml click \
    huggingface-hub tqdm hydra-core omegaconf einops munch \
    accelerate pydub tensorboard gdown
echo "Deps installed"

In [ ]:
%%bash
# Cell 4: Clone Seed-VC + install its own requirements
cd /kaggle/working/AccentShift/backend
mkdir -p third_party

if [ ! -d third_party/seed-vc ]; then
    git clone --depth 1 --filter=blob:none --single-branch \
        https://github.com/Plachtaa/seed-vc.git third_party/seed-vc
fi

# Install seed-vc deps (--no-deps avoids re-pinning torch)
pip install -q -r third_party/seed-vc/requirements.txt --no-deps 2>/dev/null || true

if [ ! -d third_party/Amphion ]; then
    git clone --depth 1 --filter=blob:none --single-branch \
        https://github.com/open-mmlab/Amphion.git third_party/Amphion
fi
echo "third_party ready"

In [ ]:
%%bash
# Cell 5: Download L2-Arctic v5.0 from Google Drive (~7GB, 1-2 min on Kaggle)
mkdir -p /kaggle/working/AccentShift/backend/data/l2arctic_raw
gdown "https://drive.google.com/uc?id=1ciCw_ttbw7a9r7d5DZzTJwoZq5rQB3TA" \
    -O /kaggle/working/AccentShift/backend/data/l2arctic_raw/l2arctic_v5.zip
echo "Download complete: $(du -sh /kaggle/working/AccentShift/backend/data/l2arctic_raw/l2arctic_v5.zip)"

In [ ]:
%%bash
# Cell 6: Unzip + organize WAVs by accent
cd /kaggle/working/AccentShift/backend
unzip -q data/l2arctic_raw/l2arctic_v5.zip -d data/l2arctic_raw/
# download_l2arctic.sh skips wget if zip already present, just organizes
bash scripts/download_l2arctic.sh
echo ""
echo "WAV counts per accent:"
for d in data/finetune/*/; do
    echo "  $(basename $d): $(find $d -name '*.wav' | wc -l) WAVs"
done

In [ ]:
%%bash
# Cell 7: TRAIN — ~2-3 hours on T4
# fp16 used (T4/P100 don't support bf16; bf16 is A100/L40S only)
# --train-ar False: CFM-only safe for 16GB T4. Add --train-ar for stronger accent (may OOM).
cd /kaggle/working/AccentShift/backend
export HF_HOME=/kaggle/working/AccentShift/backend/.hf_cache

python scripts/finetune_style.py \
    --accent all \
    --steps 15000 \
    --batch-size 8 \
    --save-every 1000 \
    --num-workers 2 \
    --mixed-precision fp16

In [ ]:
%%bash
# Cell 8: Find checkpoint + print path
find /kaggle/working/AccentShift/backend/runs/ -name '*.pth'
echo ""
echo "File sizes:"
find /kaggle/working/AccentShift/backend/runs/ -name '*.pth' -exec du -sh {} \;

In [ ]:
import shutil
# Cell 9: Zip checkpoint for download
runs_dir = '/kaggle/working/AccentShift/backend/runs/'
out_zip = '/kaggle/working/seedvc_finetuned'
shutil.make_archive(out_zip, 'zip', runs_dir)
import os
size = os.path.getsize(out_zip + '.zip') / (1024**2)
print(f'Checkpoint zipped: {out_zip}.zip ({size:.0f} MB)')
print('Download via Files panel on the right ->')